## Import Libraries

# Step 1: Install the libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Step 2: Dataset Paths

In [2]:
train_path = "dataset/train"
test_path = "dataset/test"

# Step 3: Image Processing

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

# Step 4: Load the Dataset

In [8]:
train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode="binary"
)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

Found 1243 images belonging to 2 classes.
Found 177 images belonging to 2 classes.


# Step 6: Check the Dataset

In [9]:
print(train_data.class_indices)

{'Acne': 0, 'Eczema': 1}


# Step 7: Building My Model

In [10]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),

    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

# Step 8: Compile

In [11]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Step 9: Callbacks

In [12]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model_check = ModelCheckpoint(
    "models/skin_classifier.keras",
    monitor="val_loss",
    save_best_only=True
)

# Step 10: Train the model

In [13]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=20,
    callbacks=[early_stop, model_check]
)

Epoch 1/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 304s 6s/step - accuracy: 0.8214 - loss: 0.3897 - val_accuracy: 0.8588 - val_loss: 0.3534
Epoch 2/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step - accuracy: 0.8673 - loss: 0.3012 - val_accuracy: 0.8701 - val_loss: 0.3034
Epoch 3/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 141s 4s/step - accuracy: 0.8994 - loss: 0.2535 - val_accuracy: 0.8701 - val_loss: 0.3061
Epoch 4/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 99s 3s/step - accuracy: 0.9107 - loss: 0.2146 - val_accuracy: 0.8701 - val_loss: 0.3171
Epoch 5/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.8970 - loss: 0.2241 - val_accuracy: 0.8814 - val_loss: 0.2972
Epoch 6/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.8986 - loss: 0.2257 - val_accuracy: 0.8757 - val_loss: 0.2871
Epoch 7/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 59s 2s/step - accuracy: 0.9091 - loss: 0.1986 - val_accuracy: 0.8927 - val_loss: 0.2849
Epoch 8/20
39/39 ━━━━━━━━━━━━━━━━━━━━ 63s 2s/step - accuracy: 0.9131 - loss: 0.1966 - val_accuracy: 0.8814 - val_los

# Save the model

In [14]:
model.save("models/skin_classifier.keras")

# Evaluate the Model

In [15]:
loss, accuracy = model.evaluate(test_data)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 953ms/step - accuracy: 0.8927 - loss: 0.2849
Test Loss: 0.28489628434181213
Test Accuracy: 0.8926553726196289
